# 00 – Inspect YieldSAT data (Germany)

Part A of STEP1. The German NetCDF (~7 GB) is opened lazily; features are read in pixel blocks only.
Results are collected in `sections` and written to `docs/data_overview.md` at the end.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_paths
from src.data import S2_BANDS, CATEGORICAL_VARS, FeatureReader, load_meta, open_dataset
from src import inspect_data as ins

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
paths = load_paths()
FIG_DIR = paths["docs_dir"] / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sections = {}
paths

## A1 – Files, sizes, archives

In [ ]:
trees = {}
for key in ("preprocessed", "raw"):
    trees[key] = ins.list_tree(paths[key])
    print(f"{key}: {len(trees[key])} files, {trees[key]['size_mb'].sum() / 1e3:.2f} GB")

sections["Files: preprocessed"] = trees["preprocessed"]
sections["Files: raw (per directory)"] = ins.summarize_tree(trees["raw"], depth=2)

archives = pd.concat([ins.check_archives(paths[k], trees[k]).assign(root=k) for k in trees], ignore_index=True)
sections["Archives and extraction status"] = archives
display(trees["preprocessed"])
display(sections["Files: raw (per directory)"])
display(archives)

In [ ]:
# Confirm the German file; update configs/paths.yaml if the expected path does not exist.
expected = paths["germany_nc"]
candidates = [p for p in trees["preprocessed"]["path"] if p.endswith(".nc") and "german" in p.lower()]
print("expected:", expected, "exists:", expected.exists())
print("candidates:", candidates)
sections["German NetCDF path"] = pd.DataFrame({"expected": [str(expected)], "exists": [expected.exists()],
                                               "candidates": [", ".join(candidates)]})

## A2 – Structure of the German NetCDF (lazy)

In [ ]:
ds = open_dataset(paths["germany_nc"])
dims, variables = ins.dataset_overview(ds)
sections["Dimensions"] = dims
sections["Variables"] = variables
display(dims)
display(variables)
print("global attrs:", len(ds.attrs), list(ds.attrs)[:30])
ds

## A3 – Decode categorical variables

Raw attrs are printed first so the mapping format can be checked against `src.data.get_mapping`.

In [ ]:
for v in CATEGORICAL_VARS:
    if v in ds.variables:
        a = ds[v].attrs
        print(f"--- {v} {ds[v].dims} {ds[v].dtype}: {len(a)} attrs")
        for k, val in list(a.items())[:5]:
            print(f"    {k}: {str(val)[:200]}")

meta = load_meta(ds)
print(meta.shape)
display(meta.head())
decoded = pd.DataFrame([{"variable": c, "n_unique": meta[c].nunique(),
                         "examples": ", ".join(map(str, meta[c].dropna().unique()[:6]))}
                        for c in meta.columns if not c.endswith("_code") and meta[c].dtype == object])
sections["Decoded categorical variables"] = decoded
decoded

## A4 – Fields and pixels per crop × year × farm

In [ ]:
counts = ins.counts_table(meta)
sections["Fields and pixels per crop x year x farm"] = counts
display(counts)
counts.pivot_table(index=["crop", "farm"], columns="year", values="n_fields", aggfunc="sum", fill_value=0)

## A5 – Target: field means and ground-truth attributes

In [ ]:
fields = ins.field_means(meta)
target_stats = ins.field_target_stats(fields)
sections["Field-mean yield per crop and year (t/ha)"] = target_stats
display(target_stats)
print("pixels with NaN target:", int(meta["target"].isna().sum()))

gt = ins.ground_truth_check(ds, fields)
print(f"{len(gt)} ground-truth attributes matched")
if len(gt):
    gt_summary = pd.DataFrame({
        "n_compared": [gt["target_mean"].notna().sum()],
        "n_identical_1e-4": [int(gt["identical_1e-4"].sum())],
        "max_abs_diff": [gt["diff"].abs().max()],
        "mean_diff": [gt["diff"].mean()],
    })
    sections["Pixel-mean target vs. <field>_<>_yield_ground_truth"] = gt_summary
    display(gt_summary)
    display(gt.sort_values("diff", key=abs, ascending=False).head(20))

## A6 – Inputs

### NaN fraction per timestep (S2) and NDVI per crop/year (one pass over all pixels)

In [ ]:
reader = FeatureReader(ds)
print(reader)
print("features:", reader.names)
s2 = [b for b in S2_BANDS if b in reader.names]
assert len(s2) == 12, s2

nan_frac, ndvi = ins.scan_s2(reader, meta, s2, verbose=False)
sections["S2 NaN fraction per timestep"] = nan_frac.reset_index()
nan_frac.style.format("{:.3f}").background_gradient(axis=None)

### Calendar dates per timestep (`times`) and spread between fields

In [ ]:
print(ds["times"])
times_df = ins.times_per_field(ds, meta)
times_sum = ins.times_summary(times_df)
step_days = ins.step_lengths_days(times_df)
sections["Calendar date per timestep (across field-years)"] = times_sum
print("median step length (days):", step_days)
times_sum

### Share of heuristic seeding dates

In [ ]:
seed_share = ins.seeding_type_share(meta)
sections["Seeding date type share (per field-year)"] = seed_share
seed_share

### Suspicious values: SoilGrids max ~31 216 (nodata?), `temp_*` mean ~8 225 (Kelvin sum per step?)

In [ ]:
soil = ins.soil_features(reader.names)
temp = ins.temp_features(reader.names)
print("soil:", soil)
print("temp:", temp)

soil_stats = ins.feature_stats(reader, soil) if soil else pd.DataFrame()
temp_stats = ins.feature_stats(reader, temp, step_days=step_days) if temp else pd.DataFrame()
sections["SoilGrids features (sampled blocks)"] = soil_stats
sections["Temperature features (sampled blocks)"] = temp_stats
display(soil_stats)
display(temp_stats)

In [ ]:
# Distribution of the suspicious soil maxima: is the max value a single repeated code?
if soil:
    x = reader.read(soil, 0, min(200_000, reader.n_pixels))[:, 0, :]
    for i, name in enumerate(soil):
        vals, cnt = np.unique(x[:, i][np.isfinite(x[:, i])], return_counts=True)
        top = sorted(zip(vals, cnt), key=lambda t: -t[0])[:3]
        print(f"{name:30s} top values (value, count): {top}")

## A7 – Figures

In [ ]:
rng = np.random.default_rng(0)
big = fields[fields["n_pixels"] > fields["n_pixels"].median()]
examples = list(rng.choice(big["field_year"].to_numpy(), size=3, replace=False))
ins.plot_yield_maps(meta, examples)
plt.tight_layout()
plt.savefig(FIG_DIR / "yield_maps.png", dpi=120)
plt.show()

In [ ]:
fig = ins.plot_ndvi(ndvi)
fig.savefig(FIG_DIR / "ndvi_curves.png", dpi=120)
plt.show()

## A8 – Raw / flexible format

In [ ]:
raw_tree = trees["raw"]
raw_tree["suffix"] = raw_tree["path"].map(lambda p: "".join(Path(p).suffixes).lower())
sections["Raw files per type"] = raw_tree.groupby("suffix").agg(n_files=("path", "size"),
                                                                 size_mb=("size_mb", "sum")).reset_index()
display(sections["Raw files per type"])

matches = ins.match_raw_to_fields(raw_tree, meta["field"].unique())
match_summary = matches["match"].fillna("none").value_counts().rename_axis("match").reset_index(name="n_files")
n_fields_with_raw = matches["field"].nunique()
print(f"fields in preprocessed: {meta['field'].nunique()}, fields with >=1 raw file: {n_fields_with_raw}")
sections["Raw files matched to field_shared_name"] = match_summary
display(match_summary)
display(matches.head(20))

In [ ]:
# Structure of the files belonging to one field (one file per type/modality)
example_field = matches.dropna(subset=["field"])["field"].iloc[0]
example_files = matches[matches["field"] == example_field]["path"]
print("example field:", example_field)
raw_rows = []
for rel in example_files.groupby(example_files.map(lambda p: "".join(Path(p).suffixes))).head(2):
    info = ins.describe_raw_file(paths["raw"] / rel)
    print("=" * 100)
    for k, v in info.items():
        print(f"{k}:\n{v}\n")
    raw_rows.append({"file": rel, "suffix": info["suffix"],
                     "quality/support candidates": ", ".join(info["quality_support_candidates"])})
sections[f"Raw file structure (example field {example_field})"] = pd.DataFrame(raw_rows)

## Write `docs/data_overview.md`

In [ ]:
sections["Figures"] = "![yield maps](figures/yield_maps.png)\n\n![ndvi](figures/ndvi_curves.png)"
out = paths["docs_dir"] / "data_overview.md"
ins.write_overview(sections, out)
print("wrote", out)